# Analyse exploratoire du dataset FRED — Projet Statistical Learning M1 MA Dauphine

Ce notebook présente l'analyse exploratoire (EDA) du dataset utilisé pour le projet de *Statistical Learning* du M1 Mathématiques & Applications de l'Université Paris Dauphine. L'objectif du projet est de **prédire la courbe des taux US Treasury** par régression supervisée, en comparant plusieurs méthodes (OLS, Ridge, PCR/PLS). Les données proviennent de la base [FRED](https://fred.stlouisfed.org/) (Federal Reserve Bank of St. Louis) et couvrent la période du 1ᵉʳ janvier 1990 à aujourd'hui, à la fréquence des jours ouvrés.

Le notebook s'organise en trois sections : la **Section 1** présente le contexte financier, charge le dataset et le décrit ; la **Section 2** caractérise statistiquement les trois familles de variables (taux, macro, spread) à travers des stats descriptives, des visualisations, des corrélations et une analyse de l'autocorrélation ; la **Section 3** procédera au test formel de stationnarité (Dickey-Fuller augmenté) qui tranchera définitivement les choix de preprocessing pour la modélisation.

## 1. Chargement et présentation du dataset

### 1.1 Contexte financier : qu'est-ce que la courbe des taux ?

#### Obligations du Trésor américain (*Treasuries*)

Les *Treasuries* sont des titres de dette émis par le Département du Trésor des États-Unis pour financer les dépenses publiques. Un investisseur qui achète un Treasury prête de l'argent au gouvernement américain et reçoit en échange des versements d'intérêts périodiques (appelés *coupons*) jusqu'à l'échéance, date à laquelle le principal (la valeur nominale) lui est remboursé. Parce que la probabilité de défaut de l'État américain est historiquement considérée comme quasi nulle, les Treasuries servent de référence mondiale pour le taux dit « sans risque ».

#### Maturité

La **maturité** d'une obligation est la durée qui reste jusqu'au remboursement final du principal. Les Treasuries couvrent toute la gamme : de quelques semaines (*bills*) à 30 ans (*bonds*), en passant par les *notes* de 2 à 10 ans. Dans notre dataset, nous disposons de huit points de maturité : 1 mois, 3 mois, 6 mois, 1 an, 2 ans, 5 ans, 10 ans et 30 ans.

#### Rendement (*yield*)

Le **rendement** d'une obligation est le taux d'intérêt annualisé effectif qu'un investisseur obtiendrait en la détenant jusqu'à son échéance, compte tenu du prix auquel il l'achète aujourd'hui. Ce rendement n'est pas égal au coupon affiché : il dépend du **prix de marché courant**. Plus ce prix est élevé, plus le rendement effectif est faible (relation inverse), puisque les flux futurs sont fixés et que payer plus cher pour les obtenir réduit mécaniquement la rémunération en pourcentage. Les séries FRED utilisées ici (`DGSxx`) publient directement les rendements observés chaque jour ouvré.

#### Courbe des taux (*yield curve*)

La **courbe des taux** est la fonction qui, à un instant donné, associe à chaque maturité le rendement correspondant. On la représente graphiquement avec la maturité en abscisse et le rendement en ordonnée. En régime économique « normal », cette courbe est **croissante** : les investisseurs exigent un rendement plus élevé pour prêter sur 10 ans que sur 3 mois, en compensation du risque accru (inflation, incertitude sur les taux futurs, illiquidité relative des titres longs). On parle alors d'une *courbe ascendante*. Lorsque la courbe devient **plate** ou **inversée** (taux courts supérieurs aux taux longs), c'est le signe d'un stress économique ou d'anticipations de récession.

#### Importance de la courbe

La courbe des taux est un objet central de la finance et de la macroéconomie :

- **Pricing** : elle sert de base pour évaluer toutes les obligations du secteur privé, les swaps de taux, les prêts hypothécaires à taux fixe et la plupart des produits dérivés de taux d'intérêt.
- **Signal macroéconomique** : l'inversion du spread 10 ans − 2 ans (la variable `T10Y2Y` de notre dataset) est, depuis plus de cinquante ans, l'un des meilleurs indicateurs avancés de récession aux États-Unis. Historiquement, chaque inversion durable a été suivie d'une récession dans les 12 à 24 mois.
- **Politique monétaire** : la Réserve Fédérale agit directement sur le court terme (taux des *Fed Funds*) et indirectement sur le long terme (via la *forward guidance* et les programmes d'achats d'actifs), ce qui rend la forme de la courbe révélatrice des arbitrages de politique monétaire.

#### Justification statistique du projet

Les rendements à différentes maturités sont très corrélés entre eux : quand le taux à 5 ans monte, le taux à 10 ans monte presque toujours aussi. La littérature financière (modèle de Nelson et Siegel, 1987) montre que trois facteurs latents — **niveau**, **pente** et **courbure** — expliquent typiquement plus de 99 % de la variance totale de la courbe. Cette redondance informationnelle pose un problème pour l'estimation par moindres carrés ordinaires (multicolinéarité, instabilité des coefficients, variance élevée des estimateurs) mais justifie l'usage de la **régression régularisée** (Ridge, qui pénalise les coefficients trop grands) et des **méthodes de réduction de dimension** (*Principal Component Regression*, *Partial Least Squares*), qui exploitent explicitement cette structure factorielle. Ces choix méthodologiques, qui seront au cœur du projet, trouvent ici leur justification empirique.

### 1.2 Illustration : la courbe des taux à deux dates

Avant d'entrer dans la description technique du dataset, visualisons concrètement ce que signifie « la courbe des taux » en représentant les rendements des huit maturités à deux dates contrastées :

- **1ᵉʳ juin 2007** : un an avant la crise financière, la Fed avait fortement resserré sa politique monétaire ; la courbe était *presque plate*, voire localement inversée — un signe précurseur classique.
- **1ᵉʳ juin 2010** : en pleine phase post-crise, la Fed pratiquait une politique de taux zéro et de *quantitative easing* ; la courbe est alors nettement **ascendante**, avec des taux courts proches de 0 % et un 30 ans au-dessus de 4 %.

La cellule suivante charge le dataset (nous reviendrons sur la logique de chargement en 1.3) et trace les deux courbes sur le même graphique.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Options d'affichage pandas utilisées dans tout le notebook.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_PATH = Path("..") / "data" / "raw" / "fred_data.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["date"], index_col="date")

# Correspondance ticker -> maturité en années
maturities = {
    "DGS1MO": 1 / 12,
    "DGS3MO": 3 / 12,
    "DGS6MO": 6 / 12,
    "DGS1": 1,
    "DGS2": 2,
    "DGS5": 5,
    "DGS10": 10,
    "DGS30": 30,
}
tickers = list(maturities.keys())
x = np.array(list(maturities.values()))

date_pre_crise = "2007-06-01"
date_post_qe = "2010-06-01"

y_pre = df.loc[date_pre_crise, tickers].values
y_post = df.loc[date_post_qe, tickers].values

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, y_pre, marker="o", linewidth=2, label=f"{date_pre_crise} (quasi-plate / inversée)")
ax.plot(x, y_post, marker="s", linewidth=2, label=f"{date_post_qe} (ascendante, post-QE)")
ax.axhline(0, color="grey", linewidth=0.8, alpha=0.5)
ax.set_xlabel("Maturité (années)")
ax.set_ylabel("Rendement (%)")
ax.set_title("Courbe des taux US Treasury à deux dates contrastées")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Les deux courbes illustrent deux régimes radicalement différents :

- La courbe du **1ᵉʳ juin 2007** est *presque plate* — les rendements du 3 mois au 30 ans se situent dans une fourchette étroite (environ 0,5 point de pourcentage), avec même une légère inversion sur le segment court. Ce motif reflète une politique monétaire restrictive et des anticipations d'inflation maîtrisée ; il a effectivement précédé la récession de 2008-2009.
- La courbe du **1ᵉʳ juin 2010** est *fortement ascendante* : les taux courts sont quasi nuls (effet de la politique de taux zéro de la Fed), tandis que les taux longs dépassent 4 %. L'écart 30 ans − 3 mois atteint environ 4 points, une pente très forte caractéristique d'une économie en phase de reprise avec des anticipations de remontée future des taux.

C'est précisément cette **variabilité de la forme de la courbe dans le temps** que le projet cherche à modéliser : prédire, à partir des covariables disponibles à la date `t`, la position à l'horizon d'un mois d'un point particulier de la courbe (par défaut, le rendement `DGS10`).

### 1.3 Chargement

Les données ont été téléchargées en amont par le module `src/data.py`, qui interroge l'API FRED, aligne les séries sur un calendrier de jours ouvrés et applique un forward-fill sur les variables macroéconomiques mensuelles. Le résultat est stocké dans `data/raw/fred_data.csv`. On charge ce CSV en indiquant explicitement à pandas que la colonne `date` doit être parsée comme un index temporel.

**Choix de la période 1990-01-01 → aujourd'hui**. Cette fenêtre temporelle répond à trois exigences. (1) *Disponibilité* : la majorité des séries FRED utilisées ici sont publiées sans interruption à partir des années 1990 ; remonter plus loin dans le temps créerait des trous massifs pour certaines maturités. (2) *Couverture de plusieurs régimes économiques* : la période inclut la récession de 2001 (éclatement de la bulle internet), la crise financière de 2008 et sa décennie de taux zéro, le choc COVID de 2020, puis le cycle agressif de hausses de la Fed en 2022-2024 — autant de régimes contrastés nécessaires à un apprentissage généralisable. (3) *Profondeur statistique* : avec environ 9 500 observations quotidiennes, on dispose d'une base suffisante pour réserver plusieurs années au jeu de test sans compromettre l'estimation des modèles sur le train.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt  # utilisé dans les sections suivantes
import seaborn as sns              # utilisé dans les sections suivantes

# Options d'affichage pandas : on veut voir toutes les colonnes.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Le notebook est dans notebooks/, le CSV dans data/raw/.
DATA_PATH = Path("..") / "data" / "raw" / "fred_data.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["date"], index_col="date")

Le DataFrame `df` est maintenant chargé en mémoire, indexé par date. Toutes les analyses qui suivent reposent sur ce fichier unique : toute mise à jour des données nécessite de relancer `python src/data.py` pour régénérer le CSV.

### 1.4 Aperçu du dataset

Avant toute analyse statistique, on inspecte la structure brute du DataFrame. On regarde d'abord les premières et dernières lignes (`.head()`, `.tail()`) pour vérifier la plage temporelle et l'intégrité des données, puis `.shape` pour les dimensions totales. À la place de la sortie texte classique de `.info()` — peu lisible dans un rapport — on construit un **tableau récapitulatif** qui fournit pour chaque colonne son type, le nombre de valeurs non-nulles, le nombre de NaN et le pourcentage correspondant, trié par taux de NaN décroissant.

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
# Récapitulatif par colonne : dtype, non-null, NaN, % NaN (trié décroissant)
nan_recap = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "nan": df.isna().sum(),
    "pct_nan": (df.isna().mean() * 100).round(2),
})
nan_recap.sort_values("pct_nan", ascending=False)

Le dataset contient **13 variables** observées sur **≈ 9 500 jours ouvrés**, soit environ 36 années de données (1990-01-01 à aujourd'hui). Toutes les colonnes sont de type `float64`, cohérent avec des grandeurs numériques continues ; l'index est un `DatetimeIndex`. La fréquence effective est celle des jours ouvrés (`business days`, du lundi au vendredi), choix cohérent avec les données de marché qui ne sont pas cotées les week-ends. Le tableau récapitulatif fait apparaître une hétérogénéité marquée des valeurs manquantes : la série `DGS1MO` est manquante à environ 35 % parce qu'elle n'a commencé à être publiée par le Trésor US qu'en juillet 2001 ; les autres taux affichent un taux de NaN modéré (environ 4 %) correspondant aux jours fériés américains et à quelques gaps historiques ; les variables macro mensuelles (`CPIAUCSL`, `UNRATE`, `FEDFUNDS`, `INDPRO`) et le spread `T10Y2Y` sont en pratique complets grâce au forward-fill appliqué en amont. Cette structure des valeurs manquantes sera analysée plus en profondeur en Section 2.

### 1.5 Description des variables

Le tableau ci-dessous documente la sémantique de chacune des 13 variables du dataset. On distingue deux grandes catégories :

- les **taux** (`DGSxx`) qui constituent la *courbe des taux* proprement dite, c'est-à-dire les rendements des obligations du Trésor US pour différentes maturités allant de 1 mois à 30 ans ;
- les variables **macroéconomiques** qui décrivent l'état de l'économie américaine (inflation, chômage, taux directeur, production industrielle) et un indicateur **macro dérivé** (le spread 10 ans − 2 ans).

| Ticker FRED | Description | Unité | Fréquence native | Catégorie |
|---|---|---|---|---|
| `DGS1MO` | Rendement du Treasury Bill à 1 mois *(disponible à partir de juillet 2001)* | % annuel | Quotidien | Taux |
| `DGS3MO` | Rendement à 3 mois | % annuel | Quotidien | Taux |
| `DGS6MO` | Rendement à 6 mois | % annuel | Quotidien | Taux |
| `DGS1` | Rendement à 1 an | % annuel | Quotidien | Taux |
| `DGS2` | Rendement à 2 ans | % annuel | Quotidien | Taux |
| `DGS5` | Rendement à 5 ans | % annuel | Quotidien | Taux |
| `DGS10` | Rendement à 10 ans **(variable cible candidate)** | % annuel | Quotidien | Taux |
| `DGS30` | Rendement à 30 ans | % annuel | Quotidien | Taux |
| `CPIAUCSL` | Indice des prix à la consommation (inflation) | Indice base 1982-84 = 100 | Mensuel *(forward-fill)* | Macro |
| `UNRATE` | Taux de chômage US | % | Mensuel *(forward-fill)* | Macro |
| `FEDFUNDS` | Taux effectif des Fed Funds (taux directeur) | % annuel | Mensuel *(forward-fill)* | Macro |
| `INDPRO` | Indice de production industrielle | Indice base 2017 = 100 | Mensuel *(forward-fill)* | Macro |
| `T10Y2Y` | Spread 10 ans − 2 ans (pente de la courbe) | Points de % | Quotidien | Macro dérivé |

Le *forward-fill* appliqué aux variables macro mensuelles signifie que, entre deux publications officielles, la valeur la plus récente disponible est reportée jour par jour : c'est l'information que recevrait en pratique un investisseur à la date `t`. Ce choix évite un biais de *look-ahead* (pas d'usage d'une information future pour remplir le présent) et reste fidèle au flux d'information réel des marchés.

**Usage des variables dans le projet.** Parmi ces 13 séries, `DGS10` sera la **variable cible** à prédire — choix par défaut conforme au sujet, à confirmer en phase 1. Les 12 autres variables serviront de **covariables** pour la régression. Des **features dérivées** — notamment la *pente* (différences entre maturités longues et courtes), la *courbure* (combinaison 2 × 10Y − 2Y − 30Y par exemple) et les *momenta* (variations retardées du rendement 10 ans) — seront construites dans un module de preprocessing ultérieur (`src/preprocessing.py`, non couvert dans ce notebook d'EDA). Ces features enrichissent la représentation sans introduire d'information nouvelle à strictement parler, et devront être évaluées contre la baseline OLS avant d'être conservées dans les modèles finaux.

### 1.6 Statistiques descriptives générales

La méthode `.describe()` fournit un résumé à 8 indicateurs par variable : nombre d'observations non-nulles, moyenne, écart-type, minimum, trois quartiles et maximum. On arrondit à 3 décimales pour améliorer la lisibilité sans perdre d'information significative sur des variables exprimées en pourcentages.

In [ ]:
df.describe().round(3)

Les ordres de grandeur observés sont cohérents avec la littérature macro-financière :

- **Taux Treasury** : les rendements oscillent entre un minimum proche de 0 % (période *zero-lower-bound* post-crise de 2008 et post-COVID) et un maximum d'environ 8 % atteint au début des années 1990. La moyenne des taux longs (DGS10, DGS30) est supérieure à celle des taux courts, ce qui reflète une courbe des taux *en moyenne* ascendante sur la période.
- **Taux de chômage** (`UNRATE`) : plage de 3,5 % à 14,8 %, la valeur maximale correspondant au pic de chômage d'avril 2020 provoqué par la pandémie de COVID-19.
- **Fed Funds** (`FEDFUNDS`) : plage de 0,05 % à ~8 %, avec une moyenne d'environ 2,9 %. Les longues périodes à 0,05 % correspondent à la politique de taux zéro.
- **CPI** (`CPIAUCSL`) : l'indice est monotone croissant (l'inflation cumulée ne peut pas être négative sur le long terme), ce qui explique une très large amplitude entre minimum et maximum ; on reviendra sur la nécessité de différencier cette série en Section 4.
- **Production industrielle** (`INDPRO`) : l'indice oscille autour de sa base 100 (année 2017), avec un minimum à ~60 (début des années 1990) et un maximum à ~104.
- **Spread T10Y2Y** : plage de −1,08 à +2,91 points de %. Les valeurs négatives correspondent aux épisodes d'*inversion de la courbe*, historiquement annonciateurs de récessions.

Ces observations ne sont qu'un premier aperçu : chacune sera détaillée et visualisée dans les sections suivantes, notamment les dynamiques temporelles (Section 2) et les corrélations entre maturités (Section 3).

---

**Prochaine étape** : Section 2 — Statistiques descriptives par famille de variables, visualisations temporelles et analyse des corrélations.

## 2. Statistiques descriptives par famille de variables

### 2.1 Introduction

L'objectif de cette section est de caractériser statistiquement chacune des 13 variables du dataset. Pour rendre cette analyse lisible, nous les regroupons en **trois familles sémantiques**, dont l'unification dans un seul tableau serait trompeuse :

- **Famille A — La courbe des taux Treasury** (8 variables `DGSxx`) : les objets que nous cherchons à modéliser, structurellement très corrélés entre eux ;
- **Famille B — Variables macroéconomiques** (4 variables : `CPIAUCSL`, `UNRATE`, `FEDFUNDS`, `INDPRO`) : indicateurs d'état de l'économie qui servent de covariables ;
- **Famille C — Spread dérivé** (1 variable : `T10Y2Y`) : mesure de pente, interprétée comme indicateur avancé de récession.

Pour chaque famille, nous présenterons (1) une description économique explicite de chaque variable, (2) des statistiques descriptives étendues (moyenne, écart-type, quantiles, skewness, kurtosis), (3) des visualisations temporelles et de distribution, (4) une interprétation reliée aux choix de modélisation à venir.

**Mise en garde méthodologique importante**. Plusieurs des séries étudiées (les huit taux, le CPI et la production industrielle) sont *non stationnaires* en niveau sur la période : leur moyenne et leur variance dépendent du moment où on les observe. Les statistiques descriptives calculées ici doivent donc être lues comme une **description de l'échantillon 1990-2026**, et non comme des paramètres d'une loi sous-jacente stable. Un test formel de stationnarité (ADF) est renvoyé à la Section 3, et conditionnera le choix de travailler sur les niveaux ou sur les différences dans le preprocessing.

On importe `scipy.stats` pour calculer la *skewness* (asymétrie de la distribution) et le *kurtosis* (épaisseur des queues). On définit aussi deux listes de colonnes qui structureront toute la section.

In [ ]:
from scipy import stats

# Listes de colonnes utilisees tout au long de la Section 2.
# DGS1MO est exclu : cette serie ne demarre qu'en juillet 2001 (~35% de
# NaN sur la fenetre 1990-2026), ce qui rendrait ses statistiques non
# comparables aux autres maturites. Une analyse dediee, tronquee a 2001,
# pourra la reintroduire ulterieurement.
yield_cols = ["DGS3MO", "DGS6MO", "DGS1", "DGS2",
              "DGS5", "DGS10", "DGS30"]
macro_cols = ["CPIAUCSL", "UNRATE", "FEDFUNDS", "INDPRO"]


def describe_ext(s: pd.Series) -> pd.Series:
    """Statistiques descriptives etendues pour une serie.

    Renvoie count, mean, median, std, min, Q1, Q3, IQR, max, skew, kurtosis.
    L'IQR (Q3 - Q1) est une mesure de dispersion robuste aux outliers,
    contrairement a l'ecart-type. Le kurtosis suit la convention de Fisher
    (0 pour la loi normale, > 0 si queues plus epaisses, < 0 si queues plus
    fines / distribution plus aplatie). Les NaN sont ignores.
    """
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return pd.Series({
        "count": s.count(),
        "mean": s.mean(),
        "median": s.median(),
        "std": s.std(),
        "min": s.min(),
        "Q1": q1,
        "Q3": q3,
        "IQR": q3 - q1,
        "max": s.max(),
        "skew": stats.skew(s, nan_policy="omit"),
        "kurtosis": stats.kurtosis(s, nan_policy="omit"),
    })

### 2.2 Famille A — La courbe des taux Treasury

#### 2.2.1 Description économique des maturités étudiées

**Choix de l'échantillon comparatif.** Nous excluons `DGS1MO` de cette analyse comparative car cette série n'est disponible qu'à partir de juillet 2001 (~35 % de NaN sur la fenêtre 1990-2026) ; ses statistiques ne seraient pas comparables à celles des autres maturités sur la période complète. Elle pourra être réintroduite ultérieurement après troncation à 2001. L'analyse porte donc sur **sept maturités** : `DGS3MO`, `DGS6MO`, `DGS1`, `DGS2`, `DGS5`, `DGS10`, `DGS30`.

Ces sept maturités ne jouent pas le même rôle économique ; on peut les regrouper en trois blocs avec des logiques distinctes.

- **Taux courts (`DGS3MO`, `DGS6MO`, `DGS1`, `DGS2`)** : ils reflètent principalement la *politique monétaire* de la Réserve Fédérale (taux des *Fed Funds*) et les anticipations de court terme des marchés. Ils réagissent très rapidement aux décisions du FOMC (*Federal Open Market Committee*, qui fixe le taux directeur). Leur volatilité est dominée par les cycles de resserrement / assouplissement monétaires.
- **Taux moyens (`DGS5`)** : zone de transition. Le 5 ans combine un effet politique monétaire (dans la prochaine phase du cycle) et un effet anticipations d'inflation et de croissance à moyen terme.
- **Taux longs (`DGS10`, `DGS30`)** : leur dynamique est dominée par les *anticipations d'inflation de long terme*, la *prime de terme* (prime exigée pour immobiliser de l'argent longtemps) et les flux de demande structurels (assureurs, fonds de pension, investisseurs étrangers). Ce sont eux qui servent de benchmark pour pricer les prêts hypothécaires à taux fixe, les obligations *corporate* et les swaps de taux.

La littérature financière documente que la courbe entière peut être résumée par trois facteurs latents — niveau, pente, courbure (Nelson et Siegel, 1987) — dont la PCA empirique sur nos données fournira une vérification visuelle en Section 3.

#### 2.2.2 Statistiques descriptives

On applique la fonction `describe_ext` à chaque colonne de la famille A. Chaque ligne du tableau ci-dessous correspond à une maturité, chaque colonne à un indicateur statistique.

In [ ]:
stats_yields = df[yield_cols].apply(describe_ext).T.round(3)
stats_yields

Lecture du tableau (valeurs ancrées sur la fenêtre 1990-2026) :

- **Moyennes croissantes avec la maturité** : la moyenne passe de `DGS3MO` ≈ 2,80 % à `DGS30` ≈ 4,77 %, ordonnée monotone par maturité. Cet ordre respecte la forme d'une courbe des taux « normale » en moyenne sur la période — la prime de terme est positive sur un horizon aussi long.
- **Dispersion décroissante avec la maturité** : l'écart-type passe de ≈ 2,27 sur `DGS3MO` à ≈ 1,78 sur `DGS30`. L'**IQR** (mesure robuste de dispersion, complémentaire de l'écart-type car insensible aux extrêmes) suit la même logique : 4,76 sur `DGS3MO` contre 2,75 sur `DGS30`. Les taux courts sont sensibles à chaque décision de la Fed, alors que les taux longs sont lissés par les anticipations de long terme. C'est un résultat classique de la littérature sur la structure par terme.
- **Skewness modérée et positive** : entre +0,19 (`DGS6MO`) et +0,32 (`DGS10`). Une asymétrie légère, pas dramatique, traduisant que des épisodes de taux élevés (pics > 8 %) tirent un peu la queue droite.
- **Kurtosis Fisher *négatif* sur toutes les maturités** : entre −1,25 (`DGS3MO`) et −0,60 (`DGS30`). Le kurtosis Fisher négatif (*platykurticité*) **ne traduit pas l'absence d'événements extrêmes**, mais la répartition de la masse sur plusieurs modes hétérogènes (régimes monétaires distincts), comme le confirmeront les histogrammes de la sous-section 2.2.4. Une distribution bimodale a en effet une masse concentrée sur deux pics et une queue centrale relativement plus fine qu'une gaussienne unimodale de même variance.

**Implication pour la modélisation** : les rendements Treasury en niveau ne sont pas gaussiens. Les intervalles de confiance et tests de Student de l'OLS reposent sur une hypothèse de normalité des résidus — hypothèse qu'il faudra vérifier *a posteriori* sur les résidus du modèle, et non *a priori* sur les variables brutes.

#### 2.2.3 Visualisations temporelles

Le tableau précédent donne une vue statique. On visualise maintenant l'évolution des 8 maturités sur toute la période pour faire apparaître les dynamiques communes et les éventuelles divergences.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
others = [c for c in yield_cols if c != "DGS10"]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(others)))
for col, color in zip(others, colors):
    ax.plot(df.index, df[col], label=col, color=color,
            linewidth=0.8, alpha=0.6)
ax.plot(df.index, df["DGS10"], label="DGS10",
        color="black", linewidth=2.5)
ax.set_title("Évolution des rendements Treasury par maturité "
             "(1990-aujourd'hui) — cible : DGS10 en noir")
ax.set_xlabel("Date")
ax.set_ylabel("Rendement (%)")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
          title="Maturité")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Le graphique fait ressortir plusieurs régularités fortes :

- **Tendance baissière globale** des années 1990 jusqu'à 2020 : sur notre cible `DGS10`, le maximum atteint **9,09 %** le 2 mai 1990 et le minimum **0,52 %** le 4 août 2020. Cet effondrement de quasi 9 points de pourcentage sur trente ans reflète le régime désinflationniste amorcé après la *Great Inflation* des années 1970-1980 et l'action combinée de la globalisation et de la démographie vieillissante.
- **Plateau bas 2009-2015** : après la crise financière, les taux courts restent collés à zéro sous l'effet de la *zero interest-rate policy* (ZIRP) de la Fed, et les taux longs sont écrasés par les programmes de *quantitative easing* (achats massifs d'obligations).
- **Remontée brutale 2022-2023** : face à l'inflation post-COVID, la Fed a mené l'un des cycles de hausse les plus rapides de son histoire, ce qui est visible comme un redressement quasi-vertical des taux courts.
- **Co-mouvement très fort** : les sept courbes bougent dans des directions similaires avec des timings quasi-identiques. Cette colinéarité visuelle est l'indice empirique principal qui motivera, à la sous-section 2.2.6, l'emploi de méthodes régularisées.

#### 2.2.4 Distribution des taux (Q-Q plots et histogrammes)

Au-delà des dynamiques temporelles, on regarde la distribution marginale (sans tenir compte du temps) de chaque taux à travers deux outils complémentaires :

- Les **Q-Q plots** (*quantile-quantile*) tracent les quantiles empiriques de la série contre les quantiles théoriques d'une loi normale. Si la série est gaussienne, les points s'alignent sur une droite. Une incurvation aux extrémités indique des queues plus épaisses (points au-dessus à droite, en dessous à gauche) ou plus fines que la normale (structure inverse) ; un décrochage central peut révéler une multimodalité.
- Les **histogrammes** donnent la forme fine des distributions et permettent de détecter directement une éventuelle multimodalité (plusieurs régimes de taux).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.flat, yield_cols):
    stats.probplot(df[col].dropna(), dist="norm", plot=ax)
    ax.set_title(col)
    ax.grid(True, alpha=0.3)
# 7 maturites pour 8 cases : on masque la derniere
axes.flat[-1].axis("off")
plt.suptitle("Q-Q plots des rendements Treasury contre la loi normale",
             y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.flat, yield_cols):
    df[col].hist(bins=50, ax=ax, color="steelblue", alpha=0.7)
    ax.set_title(col)
    ax.set_xlabel("Rendement (%)")
    ax.set_ylabel("Fréquence")
    ax.grid(True, alpha=0.3)
plt.suptitle("Histogrammes des rendements Treasury (bins=50)", y=1.02)
plt.tight_layout()
plt.show()

**Transition — outliers visibles dans les Q-Q plots.** Avant de synthétiser, notons que les déviations observables aux extrémités des Q-Q plots ne sont pas du bruit instrumental : elles correspondent à des **périodes de stress économique réel** — 1994 (resserrement Fed), 2008 (crise financière), 2020 (choc COVID), 2022 (remontée Fed accélérée). Nous **ne les retirons pas**, ce sont des observations économiquement valides et précieuses pour la modélisation. Leur présence renforce simplement la pertinence de **mesures robustes** (IQR, médiane, Spearman) en complément des moments classiques (moyenne, écart-type, Pearson). On peut maintenant lire les distributions en gardant cette mise en garde à l'esprit.

Les Q-Q plots révèlent un **décrochage marqué** par rapport à la première bissectrice : les points forment une courbe en S inversé, ou présentent un plateau central, plutôt qu'une droite. Les histogrammes confirment visuellement l'existence d'au moins **deux modes** sur la plupart des maturités — une concentration de masse autour de taux bas (régimes ZIRP post-2008 et post-COVID) et une autour de taux plus élevés (régime « normal » des années 1990-2000), avec des zones intermédiaires moins peuplées.

Cette **multimodalité** est le reflet statistique direct de l'existence de régimes monétaires hétérogènes. Statistiquement, elle explique le **kurtosis Fisher négatif** observé sur toutes les maturités (cf. tableau 2.2.2) : la masse n'est pas concentrée autour d'un mode unique avec des queues lourdes (cas leptokurtique), mais répartie sur plusieurs centres locaux.

**Implication pour les hypothèses de l'OLS** : une distribution marginale clairement non gaussienne sur les régresseurs n'invalide pas l'OLS en soi (les théorèmes asymptotiques portent sur les **résidus**, pas sur les régresseurs). Mais l'existence visible de régimes plaide pour vérifier soigneusement la normalité des résidus *a posteriori* (test de Shapiro-Wilk, Q-Q plot des résidus) et pour s'attendre à des intervalles de confiance OLS d'autant plus fragiles que la répartition des observations entre régimes peut biaiser localement la régression.

#### 2.2.5 Autocorrélation de la cible DGS10

**Définition rapide — Autocorrélation.** L'autocorrélation au lag $k$ d'une série $X_t$ est la corrélation de Pearson entre $X_t$ et son passé $X_{t-k}$ :

$$\rho_k \;=\; \operatorname{Corr}(X_t,\, X_{t-k}).$$

C'est l'objet central de l'analyse des séries temporelles, car il quantifie la **mémoire** de la série. Une série indépendamment et identiquement distribuée (i.i.d.) a $\rho_k = 0$ pour tout $k \geq 1$. Une série fortement persistante (quasi-marche aléatoire) a au contraire un $\rho_1$ très proche de 1 et une décroissance lente.

**Pourquoi c'est central pour notre projet.** L'OLS classique suppose que les **résidus** sont indépendants. Sur des données temporelles, cette hypothèse est rarement satisfaite, et plus l'autocorrélation des covariables est forte, plus elle se transmet potentiellement aux résidus. Surtout, l'autocorrélation forte interdit l'usage d'un k-fold naïf (qui mélange les indices temporels) : elle garantit qu'observations proches dans le temps sont quasi-redondantes, donc qu'un fold qui ne respecte pas l'ordre chronologique crée une fuite massive du futur vers le passé. C'est pourquoi le projet utilise obligatoirement `TimeSeriesSplit` (cf. CLAUDE.md).

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(df["DGS10"].dropna(), lags=60, ax=axes[0],
         title="ACF — DGS10 en niveau")
axes[0].set_xlabel("Lag (jours ouvrés)")
axes[0].grid(True, alpha=0.3)

plot_acf(df["DGS10"].diff().dropna(), lags=60, ax=axes[1],
         title="ACF — DGS10 différencié (diff première)")
axes[1].set_xlabel("Lag (jours ouvrés)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Lecture des deux graphiques.**

- **DGS10 en niveau (gauche)** : l'autocorrélation au lag 1 vaut **0,999**, au lag 5 environ **0,997**, au lag 20 environ **0,987**, et reste à **~0,957** au lag 60. Toutes les barres sortent largement de la bande de confiance à 95 %. Cette décroissance extrêmement lente est la signature typique d'un processus **quasi-marche-aléatoire** $X_t \approx X_{t-1} + \varepsilon_t$ : la valeur de demain est presque exactement celle d'aujourd'hui, plus un petit bruit. L'information « nouvelle » par jour est donc faible, et la série a une **mémoire infinie** apparente.
- **DGS10 différencié (droite)** : l'autocorrélation s'effondre. Au lag 1 elle vaut environ **+0,021**, et oscille ensuite très près de zéro pour tous les lags suivants (lag 5 ≈ −0,012, lag 60 ≈ 0,000). La quasi-totalité des barres reste dans la bande de confiance à 95 %. Le résultat est compatible avec un **bruit blanc** : les variations quotidiennes de `DGS10` sont approximativement non autocorrélées.

**Conséquences méthodologiques.**

1. La cible en niveau est non stationnaire (autocorrélation persistante), ce qui sera confirmé formellement en Section 3 par ADF. La modélisation devra probablement porter sur les **variations**, comme indiqué par CLAUDE.md.
2. Toute validation croisée doit respecter la chronologie (`TimeSeriesSplit`). Un k-fold standard sur les niveaux produirait des estimations d'erreur artificiellement optimistes, parce que les indices voisins sont quasi-redondants.
3. Sur les variations, la dépendance temporelle s'évanouit, ce qui rapproche le problème de l'hypothèse i.i.d. de l'OLS — argument supplémentaire pour préférer la modélisation en différences.

#### 2.2.6 Corrélations entre maturités

**Définition rapide — Corrélation de Pearson.** Pour deux séries $X$ et $Y$ de moyennes $\mu_X, \mu_Y$ et d'écarts-types $\sigma_X, \sigma_Y$, on définit

$$\rho(X, Y) \;=\; \frac{\operatorname{Cov}(X, Y)}{\sigma_X \, \sigma_Y} \;\in\; [-1,\, 1].$$

C'est une mesure de dépendance **linéaire** : $\rho = \pm 1$ correspond à une relation linéaire exacte. Limites importantes : (i) elle ne capte pas les relations non linéaires (par exemple $Y = X^2$ avec $X$ centré symétriquement donne $\rho = 0$), (ii) elle est sensible aux outliers, (iii) elle ne traduit pas une causalité, (iv) sur des séries non stationnaires elle peut être trompeuse (*corrélation fallacieuse*). Dans ce projet, on l'utilise pour quantifier la **multicolinéarité** entre régresseurs.

**Multicolinéarité.** On parle de multicolinéarité quand des covariables sont fortement corrélées entre elles, autrement dit quand la matrice de Gram $X^\top X$ est mal conditionnée. Cette situation a trois conséquences pour l'OLS :

1. *Instabilité numérique* de $(X^\top X)^{-1}$ : un petit changement des données entraîne une grande variation des coefficients estimés.
2. *Variance des estimateurs gonflée* : $\operatorname{Var}(\hat{\beta}) = \sigma^2 (X^\top X)^{-1}$ a des entrées explosives, donc des intervalles de confiance très larges.
3. *Coefficients individuels non interprétables* : on ne peut plus isoler l'effet marginal d'une variable.

**Trois remèdes statistiques distincts.**

- **Ridge** : ajoute une pénalité $\ell_2$ ($\lambda \, \|\beta\|_2^2$) qui rend la matrice inversible et stabilise les coefficients ; ne sélectionne **pas** de variables (toutes restent dans le modèle) ; introduit un biais qui réduit la variance.
- **Lasso** : pénalité $\ell_1$ ($\lambda \, \|\beta\|_1$) qui force certains coefficients à zéro **exactement** ; effectue donc une sélection de variables, utile quand on suspecte que peu de covariables sont vraiment pertinentes.
- **PCR** (*Principal Component Regression*) : transforme les covariables en composantes principales orthogonales, on régresse sur les premières seulement ; **atténue** la multicolinéarité par construction (les composantes sont décorrélées). Il s'agit d'une approche **non supervisée** : les composantes maximisent la variance des covariables sans tenir compte de la cible, ce qui peut sacrifier des composantes faiblement variantes mais prédictives.
- **PLS** (*Partial Least Squares*) corrige ce biais en cherchant des composantes corrélées à la cible, au prix d'une plus grande complexité de mise en œuvre.

La heatmap ci-dessous va exhiber des corrélations souvent **> 0,9** entre maturités, ce qui crée précisément le contexte où ces méthodes sont supérieures à l'OLS naïf.

In [ ]:
corr_yields = df[yield_cols].corr()
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    corr_yields, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, ax=ax, cbar_kws={"label": "Corrélation (Pearson)"},
)
ax.set_title("Matrice de corrélation — 8 maturités Treasury")
plt.tight_layout()
plt.show()

La heatmap révèle trois faits statistiques saillants, ancrés sur les valeurs effectivement calculées :

1. **Toutes les corrélations sont positives et élevées**. Le **minimum hors diagonale** vaut **0,743** (couple `DGS3MO` ↔ `DGS30`, les deux extrémités de la courbe) et le **maximum** atteint **0,998** (couple `DGS3MO` ↔ `DGS6MO`, deux taux courts adjacents). La **médiane** des corrélations hors diagonale est de **0,937** : la majorité des couples sont au-dessus de 0,9.
2. **Structure de bande** : les corrélations sont plus fortes entre maturités adjacentes (sur la diagonale et ses voisines proches) et s'affaiblissent en s'éloignant de la diagonale. Par exemple, `DGS10` ↔ `DGS30` vaut 0,985 et `DGS5` ↔ `DGS10` vaut 0,982, alors que `DGS3MO` ↔ `DGS10` tombe à 0,827. C'est la signature d'un *processus à facteurs* : quelques composantes communes expliquent l'essentiel du co-mouvement.
3. **Bloc quasi-unitaire sur les taux longs** : les corrélations entre `DGS5`, `DGS10` et `DGS30` sont toutes ≥ 0,938. Ce bloc porte l'essentiel de l'information « niveau » de la courbe.

**Conclusion méthodologique** : cette colinéarité forte est une caractéristique *intrinsèque* du problème (elle est prédite par la théorie Nelson-Siegel), pas un artefact de préparation des données. Elle **justifie empiriquement et sans ambiguïté** l'emploi de Ridge et de PCR dans le projet : une OLS naïve sur ces 7 taux comme covariables produirait des coefficients instables et numériquement problématiques.

### 2.3 Famille B — Variables macroéconomiques

#### 2.3.1 Description économique

Les quatre variables macroéconomiques retenues sont :

- **`CPIAUCSL` — *Consumer Price Index for All Urban Consumers: All Items*** : indice du niveau moyen des prix d'un panier standard de biens et services consommés par les ménages urbains américains. Attention : le CPI *lui-même* est un indice croissant ; c'est son **taux de croissance** (en glissement annuel) qu'on appelle *inflation*.

    > **Définition rapide — Équation de Fisher.** On distingue le taux d'intérêt **nominal** $i$ (observé sur le marché), le taux **réel** $r$ (ce qui rémunère vraiment le pouvoir d'achat, ajusté de l'inflation) et l'**inflation anticipée** $\pi^e$. La relation de Fisher exacte est $(1+i) = (1+r)(1+\pi^e)$, dont l'approximation au premier ordre vaut $i \approx r + \pi^e$. Cette équation explique mathématiquement pourquoi l'inflation anticipée est un déterminant direct des taux nominaux longs observés dans le dataset, et motive l'inclusion du CPI comme covariable.

- **`UNRATE` — *Unemployment Rate*** : taux de chômage mensuel publié par le Bureau of Labor Statistics. Pourquoi pertinent : la Fed a un *dual mandate* — stabilité des prix ET plein-emploi. Quand le chômage est élevé, la Fed baisse les taux pour relancer l'activité ; quand il est bas et que l'économie surchauffe, elle les remonte pour contenir l'inflation. Le chômage agit donc indirectement sur les taux courts via la fonction de réaction de la Fed.
- **`FEDFUNDS` — *Effective Federal Funds Rate*** : taux effectif au jour le jour auquel les banques se prêtent entre elles leurs réserves excédentaires. C'est la cible opérationnelle de la politique monétaire américaine : le FOMC fixe un corridor cible, et la Fed intervient sur le marché pour maintenir le taux dans ce corridor. Pourquoi pertinent : `FEDFUNDS` ancre mécaniquement les taux Treasury courts (arbitrage), donc la corrélation attendue avec `DGS3MO`, `DGS6MO` est très élevée.
- **`INDPRO` — *Industrial Production Index*** : indice mensuel du volume de production manufacturière, minière et d'utilités aux États-Unis. Pourquoi pertinent : c'est un *proxy* de la composante réelle du cycle économique. En phase d'expansion (`INDPRO` croissant), les taux longs ont tendance à monter (anticipations de resserrement monétaire et d'inflation) ; en phase de récession (`INDPRO` en baisse), ils tendent à baisser.

#### 2.3.2 Statistiques descriptives

On applique la même fonction `describe_ext` aux 4 variables macro.

In [ ]:
stats_macro = df[macro_cols].apply(describe_ext).T.round(3)
stats_macro

Ces statistiques appellent une **mise en garde forte** : `CPIAUCSL` et `INDPRO` sont des **indices quasi-monotones**. Le CPI passe de 127,5 (1990) à 326,6 (2026) avec une moyenne de 211 ; sa moyenne arithmétique et son écart-type ne décrivent **pas** un processus stable autour d'une tendance centrale, ils reflètent le fait trivial que l'indice était bas en 1990 et plus haut aujourd'hui. INDPRO va de 60,3 à 104,1 (mean 90,4, base 100 = 2017) avec une *skewness négative* (−1,11) qui traduit la longue traîne basse des années 1990. Pour ces deux variables, calculer directement une moyenne ou un écart-type n'a pas d'interprétation économique utile.

Les deux autres variables, `UNRATE` et `FEDFUNDS`, peuvent en revanche être interprétées en niveau : le chômage varie entre 3,5 % et 14,8 % avec une moyenne de 5,66 % (skew positive +1,30 reflétant les pics de récession), le taux des Fed Funds entre 0,05 % et 8,29 % avec une moyenne de 2,89 %. Ces plages sont économiquement significatives.

Pour `CPIAUCSL` et `INDPRO`, on calcule donc dans la sous-section suivante leur **taux de croissance en glissement annuel (YoY)**, transformation standard qui rend ces séries approximativement stationnaires et économiquement interprétables (inflation et croissance industrielle).

#### 2.3.3 Taux de croissance pour CPI et INDPRO

**Définition formelle — Glissement annuel (YoY).** Le taux de croissance en glissement annuel d'une variable $X_t$ avec un horizon $h$ est défini comme

$$\text{YoY}_t \;=\; 100 \times \frac{X_t - X_{t-h}}{X_{t-h}}.$$

**Choix de $h = 252$.** On utilise un décalage de **252 jours ouvrés** parce qu'il correspond au nombre approximatif de jours de cotation dans une année civile (~252 ≈ 365 × 5/7, jours fériés déduits). Comparer $X_t$ à $X_{t-252}$ revient donc à comparer deux observations distantes d'environ un an.

**Pourquoi cette transformation.** (i) Comparer la valeur à celle observée un an plus tôt **élimine les effets saisonniers** (mêmes mois comparés d'une année sur l'autre). (ii) Un indice cumulatif comme CPI ou INDPRO est non stationnaire en niveau ; le YoY produit une série *approximativement* stationnaire. (iii) L'interprétation économique est immédiate et standard : appliqué au CPI, le YoY est l'**inflation annuelle** publiée habituellement par le BLS et reprise dans les médias ; appliqué à INDPRO, c'est la **croissance annuelle de la production industrielle**.

Les 252 premières valeurs sont nécessairement NaN (pas assez d'historique pour calculer le décalage).

In [ ]:
# Taux de croissance en glissement annuel (252 jours ouvrés ≈ 1 an)
cpi_yoy = 100 * (df["CPIAUCSL"] - df["CPIAUCSL"].shift(252)) / df["CPIAUCSL"].shift(252)
indpro_yoy = 100 * (df["INDPRO"] - df["INDPRO"].shift(252)) / df["INDPRO"].shift(252)

yoy = pd.DataFrame({"CPI_YoY": cpi_yoy, "INDPRO_YoY": indpro_yoy})
yoy.apply(describe_ext).T.round(3)

Les statistiques des séries YoY sont maintenant **économiquement interprétables** :

- **Inflation CPI (YoY)** : moyenne de **2,53 %** par an, médiane **2,44 %**, très proche de la cible implicite de la Fed (2 %) ; écart-type **1,50 %**. Le minimum est **−1,96 %** atteint le 1ᵉʳ juillet 2009 (déflation ponctuelle pendant la crise financière), le maximum **+8,98 %** atteint le 1ᵉʳ juin 2022 (pic d'inflation post-COVID). Skewness positive (+1,09) qui reflète précisément ce pic 2022.
- **Croissance industrielle (YoY)** : moyenne **1,42 %**, médiane **2,24 %**, écart-type **4,08 %** (plus volatil que le CPI). Minimum **−17,41 %** le 17 avril 2020 (effondrement industriel COVID), maximum **+16,55 %** le 1ᵉʳ avril 2021 (rebond post-COVID, effet de base). Skewness fortement négative (−1,38) tirée par l'épisode récessif de 2020.

Ces moyennes et écarts-types décrivent maintenant un **processus plus proche de la stationnarité** : les transformations YoY « gomment » la tendance des indices cumulatifs et renvoient une mesure normalisée du taux de croissance. Ce sont ces séries transformées (plutôt que les niveaux bruts) qui seront utilisées dans le preprocessing final pour la modélisation.

#### 2.3.4 Visualisations temporelles

On regroupe en une grille 2×2 les quatre variables macro — `UNRATE` et `FEDFUNDS` en niveau (déjà interprétables), `CPI_YoY` et `INDPRO_YoY` en taux de croissance annuel — pour visualiser leurs dynamiques respectives sur la période.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(df.index, df["UNRATE"], color="darkred", linewidth=0.8)
axes[0, 0].set_title("Taux de chômage (UNRATE, %)")
axes[0, 0].set_ylabel("%")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(df.index, df["FEDFUNDS"], color="navy", linewidth=0.8)
axes[0, 1].set_title("Taux des Fed Funds (FEDFUNDS, %)")
axes[0, 1].set_ylabel("%")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(df.index, cpi_yoy, color="darkorange", linewidth=0.8)
axes[1, 0].axhline(2, color="grey", linestyle="--", linewidth=0.8, label="Cible Fed 2%")
axes[1, 0].set_title("Inflation CPI (YoY, %)")
axes[1, 0].set_ylabel("%")
axes[1, 0].legend(loc="upper right")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(df.index, indpro_yoy, color="darkgreen", linewidth=0.8)
axes[1, 1].axhline(0, color="grey", linestyle="--", linewidth=0.8)
axes[1, 1].set_title("Croissance production industrielle (YoY, %)")
axes[1, 1].set_ylabel("%")
axes[1, 1].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

Les quatre dynamiques racontent l'histoire macro-économique américaine récente, avec des extrêmes que l'on peut dater précisément :

- **UNRATE** : trois pics correspondent aux trois grandes récessions de la période — 2001 (bulle internet), 2008-2010 (crise financière, pic à ~10 %), 2020 (pandémie COVID, **maximum absolu à 14,8 % le 1ᵉʳ avril 2020**). Le **minimum** de la série, **3,5 %**, est atteint le 1ᵉʳ juillet 2022 — preuve que la sortie post-COVID a produit l'un des marchés du travail les plus tendus de l'histoire américaine récente.
- **FEDFUNDS** : la chronologie de la politique monétaire est lisible comme un livre. **Maximum 8,29 %** le 1ᵉʳ juin 1990 (fin de cycle de resserrement) ; effondrement après 2008, longue période ZIRP 2009-2015, normalisation lente 2016-2019, retour à 0 % en 2020 — **minimum 0,05 %** atteint le 1ᵉʳ avril 2020, soit le même mois que le pic de chômage. Cycle de hausses agressif 2022-2023 amenant le taux à ~5,5 %, la remontée la plus rapide depuis les années 1980.
- **CPI YoY** : inflation globalement sous 4 % sur 1990-2020, avec un passage en déflation à **−1,96 %** (juillet 2009). **Pic à 8,98 % en juin 2022**, qui motive précisément le resserrement agressif des Fed Funds de la même période.
- **INDPRO YoY** : croissance industrielle autour de 0-3 % en régime normal, effondrement spectaculaire en 2020 (**minimum −17,41 % le 17 avril 2020**), suivi d'un rebond rapide (**+16,55 % le 1ᵉʳ avril 2021** par effet de base).

Les quatre séries sont visiblement couplées (récessions synchronisées, réponses de politique monétaire aux chocs d'inflation ou d'emploi), ce qu'on quantifiera dans la matrice de corrélation globale de la sous-section 2.5.

### 2.4 Famille C — Spread T10Y2Y

#### 2.4.1 Interprétation économique

Le **spread T10Y2Y** est défini comme la différence entre le rendement du Treasury à 10 ans et celui du Treasury à 2 ans :

$$\text{T10Y2Y}_t \;=\; \text{DGS10}_t - \text{DGS2}_t.$$

En régime économique « normal », ce spread est **positif** : les taux longs sont supérieurs aux taux courts parce que les investisseurs exigent une prime de terme pour immobiliser leur capital plus longtemps. Quand le spread devient **négatif**, on parle d'**inversion de la courbe des taux** : une situation anormale où il est plus coûteux d'emprunter à court terme qu'à long terme. Économiquement, l'inversion révèle que les marchés anticipent une récession future — si l'économie ralentit, la Fed baissera les taux courts à un horizon de 12-24 mois, donc détenir une obligation longue au taux actuel sera rétrospectivement avantageux.

Ce phénomène a une valeur prédictive remarquable : **toutes les récessions américaines depuis 1970 ont été précédées d'une inversion durable du spread T10Y2Y, dans les 6 à 24 mois qui précèdent**. C'est l'un des rares indicateurs avancés à avoir résisté au test du temps, et c'est la raison pour laquelle la Fed de New York publie quotidiennement une probabilité de récession basée sur le spread 10Y−3M. La présence du spread dans notre dataset est donc tout sauf cosmétique : c'est un régresseur à fort pouvoir informationnel sur la dynamique future des taux.

#### 2.4.2 Statistiques et visualisation

On calcule les statistiques descriptives standard et on trace le spread dans le temps, en mettant en évidence visuellement les périodes d'inversion (zones ombrées rouges là où le spread est négatif).

In [ ]:
# Statistiques descriptives du spread
stats_t10y2y = describe_ext(df["T10Y2Y"]).to_frame("T10Y2Y").T.round(3)

# Plot temporel avec ombrage des inversions
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df.index, df["T10Y2Y"], color="steelblue", linewidth=0.8)
ax.axhline(0, color="red", linestyle="--", linewidth=1)
ax.fill_between(
    df.index, df["T10Y2Y"], 0,
    where=(df["T10Y2Y"] < 0), color="red", alpha=0.3,
    label="Inversion (spread < 0)",
)
ax.set_title("Spread T10Y2Y : différence DGS10 − DGS2 (1990-aujourd'hui)")
ax.set_xlabel("Date")
ax.set_ylabel("Spread (points de %)")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

stats_t10y2y

Le spread oscille entre **−1,08 et +2,91 points de pourcentage**, avec une **moyenne de 1,005** et une **médiane de 0,86** : une courbe ascendante est le régime par défaut sur la période. Sur l'ensemble de la série, on dénombre **1 093 jours d'inversion (11,5 % du temps)**, ce qui n'est pas anecdotique. Les zones ombrées rouges révèlent **quatre épisodes majeurs d'inversion** :

- **2000** : inversion précédant la récession de 2001 (éclatement de la bulle internet).
- **2006-2007** : inversion précédant la grande crise financière de 2008-2009, pendant environ 18 mois.
- **Été 2019** : brève inversion, suivie (de manière parfois discutée statistiquement) par la récession COVID de 2020 — même si celle-ci a été déclenchée par un choc exogène, l'inversion préalable indiquait déjà un ralentissement cyclique en cours.
- **2022-2024** : la plus longue inversion de la série, atteignant son **minimum absolu de −1,08 le 3 juillet 2023**, conséquence directe du resserrement ultra-rapide des Fed Funds face à l'inflation post-COVID. Le **maximum historique de +2,91** datait lui du 4 février 2011, en plein cycle de *quantitative easing* qui écrasait les taux courts.

La **corrélation historique entre inversion et récession ultérieure** est donc visuellement très nette. Pour le projet, cela justifie de conserver `T10Y2Y` comme covariable : bien qu'elle soit une combinaison linéaire de `DGS10` et `DGS2`, elle isole précisément la dimension « pente » qui a une valeur prédictive propre.

### 2.5 Vue d'ensemble cross-familles

#### 2.5.1 Matrice de corrélation globale

Jusqu'ici, on a regardé les trois familles séparément. Pour identifier les **ponts informationnels entre macro et courbe des taux** — qui sont précisément ce que cherche à exploiter le modèle prédictif — on calcule la matrice de corrélation complète sur les 13 variables. Les blocs de forte corrélation inter-familles sont les candidats naturels pour être des régresseurs utiles.

In [ ]:
corr_all = df.corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_all, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, ax=ax,
    cbar_kws={"label": "Corrélation (Pearson)"},
)
ax.set_title("Matrice de corrélation complète — 13 variables")
plt.tight_layout()
plt.show()

#### 2.5.2 Matrice de corrélation après transformations appropriées

**Motivation.** La matrice 2.5.1 a été calculée sur les **niveaux** des séries. Or plusieurs d'entre elles sont visiblement non stationnaires (taux Treasury, CPI, INDPRO en niveau), ce qui rend les corrélations de Pearson partiellement **fallacieuses** : deux séries qui partagent une tendance commune apparaîtront fortement corrélées même si leurs dynamiques de court terme sont indépendantes. Pour neutraliser cet artefact, on recalcule la matrice après les transformations qui rapprochent chaque série de la stationnarité :

- **Différences premières** sur les sept maturités Treasury (`DGSxx.diff()`) ;
- **Taux de croissance YoY** sur `CPIAUCSL` et `INDPRO` (transformations déjà calculées en 2.3.3) ;
- **Niveaux conservés** pour `UNRATE`, `FEDFUNDS` et `T10Y2Y` qui sont déjà des taux ou des spreads, oscillants en pratique sur la période. La stationnarité formelle de ces trois séries sera néanmoins testée en Section 3 par ADF.

In [ ]:
# Construction du DataFrame transforme
df_t = pd.DataFrame(index=df.index)
for col in yield_cols:
    df_t[col] = df[col].diff()
df_t["CPI_YoY"]    = cpi_yoy
df_t["INDPRO_YoY"] = indpro_yoy
df_t["UNRATE"]     = df["UNRATE"]
df_t["FEDFUNDS"]   = df["FEDFUNDS"]
df_t["T10Y2Y"]     = df["T10Y2Y"]

corr_transformed = df_t.corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_transformed, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, ax=ax,
    cbar_kws={"label": "Corrélation (Pearson) après transformations"},
)
ax.set_title("Matrice de corrélation après transformations "
             "(yields différenciés, CPI/INDPRO en YoY)")
plt.tight_layout()
plt.show()

**Comparaison avec la matrice en niveaux (2.5.1).** Trois constats émergent :

1. **Effondrement quasi total des corrélations FEDFUNDS ↔ yields** : la corrélation `FEDFUNDS` ↔ `DGS3MO` passe de **+0,992** (niveau) à **−0,037** (transformé). Ce résultat spectaculaire confirme directement le diagnostic de corrélation fallacieuse : la corrélation apparente entre les niveaux captait essentiellement la **tendance commune de désinflation et de baisse des taux** sur 1990-2020, pas un co-mouvement à haute fréquence. Les variations *quotidiennes* des taux courts Treasury ne sont pas linéairement liées au niveau *contemporain* du Fed Funds (qui change rarement, par décisions discrètes du FOMC).
2. **Disparition de la relation CPI ↔ taux longs** : la corrélation `CPI` ↔ `DGS10` passe de **−0,718** (niveau) à **+0,008** (CPI YoY contre `DGS10.diff()`). C'est un résultat instructif : la corrélation fortement négative observée en 2.5.1 n'avait **rien** d'une relation économique stable, c'était uniquement le produit de deux tendances qui s'accordaient sur la fenêtre 1990-2020 (CPI monotone croissant, taux longs en tendance baissière). Une fois les deux séries ramenées à leur composante stationnaire, la relation linéaire instantanée disparaît presque entièrement. Cela ne veut pas dire que l'inflation n'a pas d'effet sur les taux — l'équation de Fisher reste valide — mais simplement que cet effet ne se lit pas dans la corrélation contemporaine quotidienne ; il opère à des horizons plus longs et avec des effets retardés que la corrélation simple ne capture pas.
3. **Conservation de la colinéarité intra-yields, mais atténuée** : `DGS10.diff()` ↔ `DGS30.diff()` reste à 0,916 (contre 0,985 en niveau) ; `DGS3MO.diff()` ↔ `DGS30.diff()` tombe à 0,194 (contre 0,743). La structure de bande persiste, mais avec un écart bien plus marqué entre extrémités proches et extrémités éloignées de la courbe — la « fenêtre » de co-variation à haute fréquence est plus étroite que ne le suggérait l'analyse en niveaux. La motivation pour Ridge / PCR reste valable mais la multicolinéarité effective du modèle préprocessé sera moins extrême qu'en analyse en niveaux.

**Conclusion.** Cette comparaison illustre concrètement pourquoi le **preprocessing en différences / YoY est indispensable** avant la modélisation : non seulement il rend les hypothèses statistiques de l'OLS plus défendables, mais il transforme aussi le diagnostic de relations entre variables. Plusieurs liens « évidents » en niveau s'évanouissent en stationnaire, et inversement, certaines relations subtiles (par exemple entre les variations de `CPI_YoY`, `UNRATE` et `FEDFUNDS` qui présentent des corrélations modérées non nulles dans cette matrice transformée) deviennent visibles.

Quatre structures ressortent de la matrice complète :

1. **Bloc intra-taux (7 × 7, coin supérieur gauche)** : déjà analysé en 2.2.6, corrélations très élevées (de 0,743 à 0,998), structure de bande.
2. **FEDFUNDS ↔ taux courts** : corrélation très forte. On mesure **0,992** sur `FEDFUNDS` ↔ `DGS3MO`, **0,989** sur `DGS6MO`, **0,980** sur `DGS1`. C'est la signature directe de la transmission de la politique monétaire : les taux courts Treasury sont mécaniquement ancrés au taux des Fed Funds par arbitrage. Cette redondance signifie que `FEDFUNDS` apporte peu d'information supplémentaire aux taux courts — à garder en tête pour éviter de sur-pondérer cette dimension dans la régression.
3. **CPI en niveau ↔ taux** : la matrice donne des corrélations **fortement négatives** (`CPI` ↔ `DGS10` = **−0,718**, ↔ `DGS30` = **−0,762**), résultat qui peut sembler contre-intuitif au regard de l'équation de Fisher. La corrélation de Pearson **n'a pas d'interprétation économique stable** ici : Pearson mesure une corrélation entre **réalisations**, pas entre processus, et son signe et sa magnitude entre une série non stationnaire (CPI en niveau, monotone croissant) et une série oscillante (taux) **dépendent de la fenêtre temporelle considérée**. La sous-section 2.5.2 recalcule cette corrélation après transformation appropriée et donne le verdict.
4. **T10Y2Y et structure de pente**. `T10Y2Y` est par construction une combinaison linéaire de `DGS10` et `DGS2`, donc l'absence de corrélation avec `DGS10` est numérique et non structurelle. La corrélation observée avec `DGS10` (**−0,155**) est numériquement faible, mais cela s'accompagne d'une corrélation **modérée** avec `DGS2` (**−0,536**) et même **forte** avec `DGS3MO` (**−0,649**). `T10Y2Y` apporte donc principalement une information de **pente** complémentaire au court de la courbe, pas une dimension orthogonale absolue. À noter aussi sa corrélation positive forte avec `UNRATE` (**+0,705**) : les inversions précèdent les récessions et donc les phases de chômage croissant.

Ces observations renforcent trois décisions méthodologiques : (a) **régulariser** (Ridge ou PCR) vu la redondance intra-taux et FEDFUNDS-taux courts, (b) **travailler sur les différences ou taux de croissance** pour CPI et INDPRO afin de révéler leur vraie relation avec les taux, (c) **standardiser** les covariables avant toute régularisation (Ridge pénalise les coefficients en norme $\ell_2$, ce qui n'a pas de sens si les variables ont des échelles hétérogènes — pourcentages, indices à niveaux différents, points de base).

### 2.6 Synthèse de la Section 2

**Sur les distributions.** Les rendements Treasury en niveau ne sont pas gaussiens : skewness **modérée et positive** (entre +0,19 et +0,32 selon les maturités), **kurtosis Fisher négatif** sur toutes les maturités (de −1,25 à −0,60), et surtout **multimodalité** marquée liée à l'existence de régimes monétaires hétérogènes (pré/post-2008, pré/post-COVID). Le kurtosis négatif (platykurticité) ne signifie *pas* l'absence d'événements extrêmes mais traduit précisément la répartition de la masse sur plusieurs modes plutôt que sur un mode unique à queues lourdes. Les ordres de grandeur sont physiquement cohérents : taux Treasury de 0,52 % à 9,09 %, chômage de 3,5 % à 14,8 %, inflation typiquement autour de 2,5 % (moyenne) avec un pic à 8,98 % en juin 2022. Les variables d'indice (CPI, INDPRO) ne sont pas analysables en niveau ; leur transformation en taux de croissance annuel les rend statistiquement exploitables.

**Sur les relations entre variables.** La matrice de corrélation fait apparaître plusieurs blocs structurels : (1) une **très forte colinéarité intra-taux** (médiane 0,937, max 0,998) résultat direct de la structure factorielle de la courbe, (2) un **couplage mécanique FEDFUNDS–taux courts** (~0,99) reflétant la transmission de la politique monétaire, (3) une corrélation `T10Y2Y`–taux **asymétrique** (faible avec `DGS10`, modérée avec `DGS2`-`DGS3MO`) qui isole une information de pente partiellement complémentaire, (4) une corrélation `T10Y2Y` ↔ `UNRATE` de +0,705, signature de la valeur prédictive du spread sur le cycle économique.

**Sur la dépendance temporelle.** L'analyse d'autocorrélation (2.2.5) révèle que `DGS10` en niveau est **extrêmement persistant** (ACF lag 1 = 0,999, lag 60 ≈ 0,96), ce qui est typique d'un processus quasi-marche-aléatoire. Après différenciation, l'autocorrélation s'effondre quasi à zéro (lag 1 ≈ 0,02), ce qui indique un comportement compatible avec un bruit blanc. Cette structure rend obligatoire le recours à une validation croisée **temporelle** (`TimeSeriesSplit`) : un k-fold standard ré-échantillonnerait les indices et créerait une fuite massive du futur vers le passé.

**Ce qu'il reste à établir.** Les caractères *non stationnaires* des taux en niveau, du CPI et d'INDPRO sont visibles graphiquement et leur autocorrélation persistante en est un indice fort, mais n'ont pas été **testés formellement**. C'est l'objet de la Section 3 (test de Dickey-Fuller augmenté, ADF), qui tranchera entre modélisation en niveaux, en différences premières ou en taux de croissance. Les résidus des modèles de régression devront aussi être analysés *a posteriori* pour vérifier les hypothèses classiques de l'OLS (normalité, homoscédasticité, non-autocorrélation), ce qui se fera en phase de validation.

**Conséquences pour la modélisation.** Trois choix méthodologiques sont déjà justifiés par l'analyse descriptive seule, sans attendre les tests : (1) **régularisation** (Ridge) ou **réduction de dimension** (PCR, PLS) plutôt qu'OLS naïf sur les 7 taux, en raison de la multicolinéarité ; (2) **standardisation** systématique des covariables dans le pipeline, pour que la pénalité Ridge ait du sens sur des variables à échelles différentes ; (3) probable usage des **différences premières** ou des **taux de croissance YoY** pour les variables non stationnaires en niveau, à confirmer par ADF.

---

**Prochaine étape** : Section 3 — Test formel de stationnarité (Dickey-Fuller augmenté, ADF) et décisions finales de preprocessing pour la modélisation.